# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exoxeph/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook continues my Week 1 choice: **Lane 2 — Refresh / Content Opportunity Scoring**. I am framing it as a real ML decision-support task before training a model.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is mainly a **ranking/scoring task**. Each eligible content page should receive a priority score and the pages should be sorted from highest to lowest so a content or SEO reviewer can inspect the strongest refresh opportunities first. I may use a supervised classifier internally to estimate the likelihood of observed decline, but the useful product is not a hard yes/no decision. It is a **ranked review queue** that helps a human decide where limited review time should go first.

In [1]:
# Load the same starter data used in Week 1.
# This works from the repo root, work/notebooks/, or a fresh Colab session.
from pathlib import Path
import subprocess
import pandas as pd
import numpy as np

DATA_REL = Path("data/raw/content_refresh_anonymized.csv")

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]

data_path = next(
    (root / DATA_REL for root in candidate_roots if (root / DATA_REL).exists()),
    None,
)

if data_path is None:
    repo_dir = Path("/content/flyrank-ml-internship-starter")
    if not repo_dir.exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                str(repo_dir),
            ],
            check=True,
        )
    data_path = repo_dir / DATA_REL

if not data_path.exists():
    raise FileNotFoundError(f"Could not find starter data at {data_path}")

df = pd.read_csv(data_path)

df["is_declining_label"] = (
    df["trend_direction"].fillna("").str.lower().eq("down").astype(int)
)

eligible = (
    df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
    .drop_duplicates("content_id")
    .copy()
)

print(f"Eligible content items: {len(eligible):,}")
print(f"Unique content_ids: {eligible['content_id'].nunique():,}")
print(f"Observed declining share: {eligible['is_declining_label'].mean():.1%}")
print("Decision output: one priority score per content item -> ranked human-review queue")

Eligible content items: 30,000
Unique content_ids: 30,000
Observed declining share: 54.2%
Decision output: one priority score per content item -> ranked human-review queue


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My working target is **`is_declining_label`**, where 1 means the page's observed `trend_direction` is `down` and 0 means it is not. I am using this as a **proxy for refresh opportunity**, not as proof that the page must be refreshed. The label is derived from an observed performance trend in the starter snapshot, while my own review rules do not define the label.

Because `trend_direction` is computed from `trend_pct`, both are direct label information and must **not** be model features. I will also exclude product flags such as `health_score`, `is_quick_win`, or `is_initial_refresh_candidate` when they are present because they can encode the same decision logic. A later version of the project should use a genuinely future outcome window; for this Week 2 framing, the starter snapshot label is only a practical proxy.

In [2]:
# Check the proxy target and explicitly define features that are allowed at this stage.
candidate_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "content_type",
    "main_intent",
]

leakage_or_product_columns = [
    "trend_direction",
    "trend_pct",
    "health_score",
    "needs_indexing",
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "ai_opportunity",
    "is_underperformer",
    "is_declining",
    "is_initial_refresh_candidate",
]

present_features = [c for c in candidate_features if c in eligible.columns]
present_leakage = [c for c in leakage_or_product_columns if c in eligible.columns]

target_counts = eligible["is_declining_label"].value_counts().sort_index()
print("Target counts:")
print(target_counts.rename(index={0: "not_down", 1: "down"}))
print()
print(f"Candidate non-label features available: {len(present_features)}")
print("Explicitly excluded label/product columns present in this data:")
print(present_leakage)

assert "trend_direction" not in present_features
assert "trend_pct" not in present_features

Target counts:
is_declining_label
not_down    13738
down        16262
Name: count, dtype: int64

Candidate non-label features available: 12
Explicitly excluded label/product columns present in this data:
['trend_direction', 'trend_pct']


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary metric will be **Precision@50** on a held-out set of clients. It asks: among the 50 pages the system ranks highest for review, what fraction actually have the observed declining proxy? This matches the real action because a reviewer has a limited queue rather than unlimited time.

I will call the ranking **good** if **Precision@50 is at least 0.70** and it also beats the transparent fixed-rule baseline on the same held-out pages. A value of 0.70 means at least **35 of the first 50** recommendations are observed decliners. The comparison must be made on held-out clients, not on the rows used to fit the model.

In [3]:
# Make the metric concrete before training anything.
K = 50
GOOD_PRECISION_AT_K = 0.70
base_rate = eligible["is_declining_label"].mean()


def precision_at_k(frame, score_col, target_col="is_declining_label", k=50):
    top = frame.sort_values(score_col, ascending=False).head(k)
    return float(top[target_col].mean())

print(f"Overall observed declining rate: {base_rate:.3f}")
print(
    f"Success target: Precision@{K} >= {GOOD_PRECISION_AT_K:.2f} "
    f"= at least {int(K * GOOD_PRECISION_AT_K)} of {K} top-ranked pages declining"
)
print(f"Required lift over the overall base rate: {GOOD_PRECISION_AT_K - base_rate:+.3f}")

# A tiny sanity baseline: rank only by visibility.
# This is NOT the final FlyRank hand-rule baseline; it only proves the metric is computable now.
metric_check = eligible.copy()
metric_check["visibility_only_score"] = np.log1p(metric_check["impressions_90d"])
visibility_p50 = precision_at_k(metric_check, "visibility_only_score", k=K)
print(f"Visibility-only sanity baseline Precision@{K}: {visibility_p50:.3f}")

Overall observed declining rate: 0.542
Success target: Precision@50 >= 0.70 = at least 35 of 50 top-ranked pages declining
Required lift over the overall base rate: +0.158
Visibility-only sanity baseline Precision@50: 0.420


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one eligible pseudonymized content item/page.** I keep pages with at least one impression, require `content_age_days >= 90`, and deduplicate on `content_id`, matching the Week 1 starter logic. The row contains observed search, engagement, freshness, and content signals plus the decline proxy. The eventual output for each row is a review priority score; a human reviewer then decides whether any actual content action is needed.

In [4]:
# Build and display the lane-specific dataframe.
display_columns = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "is_declining_label",
]
display_columns = [c for c in display_columns if c in eligible.columns]

lane_df = eligible[display_columns].copy()

print(f"Lane dataframe shape: {lane_df.shape}")
print(f"One row per content item? {lane_df['content_id'].is_unique}")
assert lane_df["content_id"].is_unique

display(lane_df.head(8))

Lane dataframe shape: (30000, 14)
One row per content item? True


,content_id,client_id,content_type,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,17,10.6,0.76,5.88,4.55,187,20,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,9,20.3,0.05,0.00,10.00,445,25,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,11,36.5,0.09,0.00,28.57,141,20,3515.0,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,78,6.2,0.49,1.28,3.45,463,22,NaN,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,145,44.0,0.13,0.00,24.29,263,14,2803.0,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,5,8.5,0.03,0.00,25.00,147,20,3080.0,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,1,7.0,0.00,0.00,0.00,90,20,3059.0,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,28,21.2,0.06,3.57,7.14,445,22,NaN,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule would force me to choose one or two thresholds such as “old page + high impressions” or “low CTR + visible page.” The real priority depends on several signals at the same time. For example, the same CTR can mean something different at position 2 versus position 25, and an old page with 50 impressions should not necessarily outrank a moderately old page with thousands of impressions. Visibility, freshness, position, engagement, and content characteristics can interact rather than follow one clean threshold.

ML therefore earns its place only if it learns those combinations well enough to rank held-out pages better than the simple rules. I will still keep the fixed-rule baseline as the comparison; if ML does not improve the review queue, the simpler rule should win.

In [5]:
# Compare a few simple if-statements to show their precision/coverage trade-off.
# These are diagnostics, not the final model or final baseline.
rules = {
    "visible: impressions >= 500": eligible["impressions_90d"] >= 500,
    "stale: last update >= 180d": eligible["days_since_last_update"] >= 180,
    "visible + stale": (
        (eligible["impressions_90d"] >= 500)
        & (eligible["days_since_last_update"] >= 180)
    ),
    "visible + low CTR (<0.5%)": (
        (eligible["impressions_90d"] >= 500)
        & (eligible["ctr"] < 0.5)
    ),
    "visible + position 1-20": (
        (eligible["impressions_90d"] >= 500)
        & (eligible["avg_position"] > 0)
        & (eligible["avg_position"] <= 20)
    ),
}

total_decliners = int(eligible["is_declining_label"].sum())
summary_rows = []

for rule_name, mask in rules.items():
    flagged = eligible.loc[mask]
    declining_flagged = int(flagged["is_declining_label"].sum())
    summary_rows.append(
        {
            "rule": rule_name,
            "pages_flagged": len(flagged),
            "declining_rate_among_flagged": (
                declining_flagged / len(flagged) if len(flagged) else np.nan
            ),
            "share_of_all_decliners_captured": (
                declining_flagged / total_decliners if total_decliners else np.nan
            ),
        }
    )

rule_summary = pd.DataFrame(summary_rows)
display(
    rule_summary.style.format(
        {
            "declining_rate_among_flagged": "{:.1%}",
            "share_of_all_decliners_captured": "{:.1%}",
        }
    )
)

print(
    "No single threshold is automatically best: rules trade off how many pages they flag "
    "against how concentrated the observed decline is."
)

,rule,pages_flagged,declining_rate_among_flagged,share_of_all_decliners_captured
0,visible: impressions >= 500,16726,59.6%,61.3%
1,stale: last update >= 180d,174,47.1%,0.5%
2,visible + stale,17,94.1%,0.1%
3,visible + low CTR (<0.5%),14245,61.5%,53.9%
4,visible + position 1-20,12023,59.9%,44.3%


No single threshold is automatically best: rules trade off how many pages they flag against how concentrated the observed decline is.


## Self-check

Before I submit:

- [x] Every section above is filled — markdown thinking AND code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries are included
- [x] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/`, then submit the repo URL on the card

The two unchecked items are execution/submission actions I must complete in Colab/GitHub.